# Pivot_table, groupby

equivalence between two methods:
- `df.pivot_table(index='country', columns='state', values='gdp', aggfunc='mean')`
- `df.groupby(['country','state'])['gdp'].mean().unstack()`

 | Method | Description |
|---|---|
| `pd.pivot_table(values=None, index=None, columns=None, aggfunc='mean', fill_value=None, margins=False, margins_name='All', dropna=True, observed=False, sort=True)` | Create a pivot table. `index` and `columns` accept Series, column names, `pd.Grouper`, or lists. `aggfunc` can be a function, list, or dict mapping columns to functions. Missing values replaced with `fill_value`. `margins=True` adds subtotals/totals. `dropna=False` keeps empty column combinations. `observed=True` shows only observed categories for categorical groupers. |

In [50]:
from IPython.display import display
import numpy as np
import pandas as pd

df = pd.DataFrame({'age':[15, 17, 10],'growth':[.5, .7, 1.2],'Name':['Paul', 'George', 'Ringo'] })
display(df.columns)
display(df.index)
display(df.info())


df0 = pd.DataFrame({
    "firm":    ["A", "A", "A", "A", "B", "B", "B", "B", "C", "C", "C", "C"],
    "year":    [2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023],
    "region":  ["East", "East", "East", "East", "West", "West", "West", "West", "East", "East", "East", "East"],
    "sales":   [120, 135, 135, 160, 90, 110, 105, 140, 120, np.nan, 130, 130],
    "profit":  [15, 18, 18, 25, 10, 12, 11, 20, 15, 16, 19, 19]
}).rename(index={i: f'id-{i}' for i in range(12) })

#What is the average profit by region for each firm?
#- Put the region in the index
#- Have a column for each firm
#- Put the average profit in each cell

display(df0.pivot_table(index='region', columns='firm',values='profit', aggfunc='mean'))

display(df0.pivot_table(index='region', columns='firm',values='profit', aggfunc='mean', margins=True))

Index(['age', 'growth', 'Name'], dtype='str')

RangeIndex(start=0, stop=3, step=1)

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     3 non-null      int64  
 1   growth  3 non-null      float64
 2   Name    3 non-null      str    
dtypes: float64(1), int64(1), str(1)
memory usage: 219.0 bytes


None

firm,A,B,C
region,,,
East,19.0,NaN,17.25
West,NaN,13.25,NaN


firm,A,B,C,All
region,,,,
East,19.0,NaN,17.25,18.125
West,NaN,13.25,NaN,13.250
All,19.0,13.25,17.25,16.500


In [36]:
df0.groupby(['region','firm'])['profit'].mean().unstack()

firm,A,B,C
region,,,
East,19.0,NaN,17.25
West,NaN,13.25,NaN


In [16]:
df0.pivot_table(index='region', columns='firm',values='profit', aggfunc='mean').reset_index().rename_axis(columns=None)

,region,A,B,C
0,East,19.0,NaN,17.25
1,West,NaN,13.25,NaN


In [9]:
df0.pivot_table(index='region', columns='firm',values='profit', aggfunc='mean').unstack()

firm  region
A     East      19.00
      West        NaN
B     East        NaN
      West      13.25
C     East      17.25
      West        NaN
dtype: float64

In [30]:
# .unstack() convert a multi index series to data frame
display(df0.groupby(['firm', 'region']).profit.mean().round(2))
display('------')
# row index name: firm
# column index name: region
display(df0.groupby(['firm', 'region']).profit.mean().round(2).unstack().head(3))
display(' firm is not a colunm, row index name: firm; column index name: region')
display(df0.groupby(['firm', 'region']).profit.mean().round(2).unstack().columns)
display(df0.groupby(['firm', 'region']).profit.mean().round(2).unstack().index)

firm  region
A     East      19.00
B     West      13.25
C     East      17.25
Name: profit, dtype: float64

'------'

region,East,West
firm,,
A,19.00,NaN
B,NaN,13.25
C,17.25,NaN


' firm is not a colunm, row index name: firm; column index name: region'

Index(['East', 'West'], dtype='str', name='region')

Index(['A', 'B', 'C'], dtype='str', name='firm')

In [27]:
display(df0.groupby(['firm', 'region']).profit.mean().round(2).unstack().reset_index().head(3))
display(df0.groupby(['firm', 'region']).profit.mean().round(2).unstack().reset_index().columns)
display('region is the column index name')


region,firm,East,West
0,A,19.00,NaN
1,B,NaN,13.25
2,C,17.25,NaN


Index(['firm', 'East', 'West'], dtype='str', name='region')

'region is the column index name'

In [28]:
display(df0.groupby(['firm', 'region']).profit.mean().round(2).unstack().reset_index().rename_axis(columns=None).head(3))

,firm,East,West
0,A,19.00,NaN
1,B,NaN,13.25
2,C,17.25,NaN


# pd.get_dummies

converting categorical (don't need to be `astype('cat')`) into dummies.

- `.merge(how='left', left_on=True, right_on=True)` <=> `.join()`

In [40]:
df0.pipe(lambda df_: df_.merge(pd.get_dummies(df_['firm'], prefix='firm', drop_first=True, dtype=int ), how='left', left_index=True, right_index=True))

,firm,year,region,sales,profit,firm_B,firm_C
id-0,A,2020,East,120.0,15,0,0
id-1,A,2021,East,135.0,18,0,0
id-2,A,2022,East,135.0,18,0,0
id-3,A,2023,East,160.0,25,0,0
id-4,B,2020,West,90.0,10,1,0
id-5,B,2021,West,110.0,12,1,0
id-6,B,2022,West,105.0,11,1,0
id-7,B,2023,West,140.0,20,1,0
id-8,C,2020,East,120.0,15,0,1
id-9,C,2021,East,NaN,16,0,1


# Custom Aggregation Function
| Method | Description |
|---|---|
| `.groupby(by=None, axis=0, level=None, as_index=True, sort=True, group_keys=True, observed=False, dropna=True)` | Return a `GroupBy` object grouped by `by` (column name, function, Series, `pd.Grouper`, or list). Use `as_index=False` to keep grouping keys as columns. `observed=True` only shows observed categories for categorical groupers. `dropna=False` keeps groups with no values. |
| `.stack(level=-1, dropna=True)` | Push a column level into the index. `level` selects the column level to stack (`-1` innermost). |
| `.unstack(level=-1, dropna=True)` | Push an index level into the columns. `level` selects the index level to unstack (`-1` innermost). |

In [ ]:
# what is the percent of A firm in each region?
def per_emacs(ser):
    return ser.str.contains('A').sum() / len(ser) * 100

df0.pivot_table(index='region', values='firm', aggfunc=per_emacs)

,firm
region,
East,50.0
West,0.0


In [44]:
df0.groupby('region')[['firm']].agg(per_emacs)

,firm
region,
East,50.0
West,0.0


# multiple aggregates


In [45]:
df0.pivot_table(index='region', values='profit', aggfunc=('min', 'max')).head(3)

,max,min
region,,
East,25,15
West,20,10


In [47]:
df0.groupby(['region']).profit.agg(['min','max']).head(3)

,min,max
region,,
East,15,25
West,10,20


# Per Column Aggregations

- `agg(func)=('min','max')` for `groupby(pivot_table)`
    - note: `aggfunc=('min', 'max')` gives slightly different results
- `agg(func)={'profit':['min','max]}`
- `agg(sale_min=('sales', 'min'))` new name

> `pd.IndexSlice[]` to access specific label

In [ ]:
col_list=['profit', 'sales']
df0.pivot_table(index='region', values=col_list,aggfunc=('min', 'max'))

min           max       
       profit  sales profit  sales
region                            
East       15  120.0     25  160.0
West       10   90.0     20  140.0

In [60]:
df0.groupby('region')[['profit','sales']].agg(('min','max'))

profit      sales       
          min max    min    max
region                         
East       15  25  120.0  160.0
West       10  20   90.0  140.0

In [ ]:
# What are the minimum and maximum profit and the average sales for region?

jb3=(df0
     .groupby('region')
     .agg({'profit': ['min', 'max'], 'sales': 'max'})
     .head(3))
display(jb3.columns) # columns have multi index
display(jb3[('profit', 'min')] ) # access the multi index
display(jb3['sales']['max'])    # access the multi index
display(jb3.loc[:, pd.IndexSlice[:, 'max']])

MultiIndex([('profit', 'min'),
            ('profit', 'max'),
            ( 'sales', 'max')],
           )

region
East    15
West    10
Name: (profit, min), dtype: int64

region
East    160.0
West    140.0
Name: max, dtype: float64

,profit,sales
,max,max
region,,
East,25,160.0
West,20,140.0


In [57]:
# identical
df0.pivot_table(index='region',aggfunc={'profit': ['min', 'max'],'sales': 'max'}).head(3)

profit      sales
          max min    max
region                  
East       25  15  160.0
West       20  10  140.0

>> customized name cannot be done in `pivot_table()`. Use a keyword parameter to pass in a tuple of the column and aggregation function. The keyword parameter will be turned into a (flattened) column name.

In [56]:
df0.groupby('region', observed=True).agg(sale_min=('sales', 'min'),profit_max=('profit', 'max')).head(3)

,sale_min,profit_max
region,,
East,120.0,25
West,90.0,20


In [49]:
import pandas as pd

df_cntry = pd.DataFrame({
    'country_live': pd.Categorical(
        ['US', 'US', 'CA'],
        categories=['US', 'CA', 'UK']   # UK exists as category, but no rows
    ),
    'age': [20, 30, 40]
})

# observed=False (include all category levels)
display(df_cntry.groupby('country_live', observed=False).agg(age_min=('age', 'min')))
# observed=True (only categories that appear in data)
display(df_cntry.groupby('country_live', observed=True).agg(age_min=('age', 'min')))

,age_min
country_live,
US,20.0
CA,40.0
UK,NaN


,age_min
country_live,
US,20
CA,40


# Grouping by Hierarchy

In [65]:
df0.pivot_table(index=['region', 'firm'],values='profit', aggfunc=['min', 'max'], observed=True).head(3)

min    max
            profit profit
region firm              
East   A        15     25
       C        15     19
West   B        10     20

In [66]:
df0.groupby(['region', 'firm'],observed=True)[['profit']].agg('mean').reset_index().head(3)

,region,firm,profit
0,East,A,19.00
1,East,C,17.25
2,West,B,13.25


# group by function

| Method | Description |
|---|---|
| Column access | Access a column by attribute or index operation. |
| `g.agg(func=None, *args, engine=None, engine_kwargs=None, **kwargs)` | Apply an aggregate `func` to groups. `func` can be a string, function (accepts a column and returns a reduction), a list, or a dict mapping column name to string/function/list. |
| `g.all(skipna=True)` | Collapse each group to `True` if all values are truthy (per group). |
| `g.any(skipna=True)` | Collapse each group to `True` if any values are truthy. |
| `g.apply(func, *args, **kwargs)` | Apply a function to each group. Function accepts the group (DataFrame) and returns a scalar, Series, or DataFrame. Returns Series, DataFrame (each Series as a row), or DataFrame (with group index as inner index), respectively. |
| `g.count()` | Count of non-missing values for each group. |
| `g.max(numeric_only=False, min_count=-1)` | Return the maximum row of each group. `min_count>0` fills insufficient groups with `NaN`. |
| `g.mean(numeric_only=True)` | Return the mean of each group. |
| `g.min(numeric_only=False, min_count=-1)` | Return the minimum row of each group. `min_count>0` fills insufficient groups with `NaN`. |
| `g.pipe(func, *args, **kwargs)` | Apply `func` to each group (functional chaining). |
| `g.quantile(q=.5, interpolation='linear')` | Return quantile(s) for each group; pass list `q` to get inner index for each quantile. |
| `g.rank(method='average', na_option='keep', ascending=True, pct=False, axis=0)` | Return numerical ranks within each group. `method` handles ties (`'average'`, `'min'`, `'max'`, `'first'`, `'dense'`). `na_option` controls NaN handling (`'keep'`, `'top'`, `'bottom'`). |
| `g.resample(rule, *args, **kwargs)` | Create a resample object with time-frequency `rule`. Needs further aggregation. |
| `g.rolling(window_size)` | Create a rolling grouper. Needs further aggregation. |
| `g.sample(n=None, frac=None, replace=False, weights=None, random_state=None)` | Return a sample from each group (keeps original index). |
| `g.shift(periods=1, freq=None, axis=0, fill_value=None)` | Return shifted values for each group (keeps original index). |
| `g.size()` | Return a Series with the size (row count) of each group. |
| `g.sum(numeric_only=True, min_count=0)` | Return sum for each group. |


In [73]:
def even_grouper(idx):
    return 'odd' if idx % 2 else 'even'

df.pivot_table(index=even_grouper, aggfunc='size')

even    2
odd     1
dtype: int64

In [74]:
df.groupby(even_grouper).size()

even    2
odd     1
dtype: int64

# filtering parts of groups

If you want to filter based on aggregated data but keep the original index (sans filtered rows), use the `filter` method on the groupby object.

| Method | Description |
|---|---|
| `g.filter(func, dropna=True, *args, **kwargs)` | Return the original DataFrame but with filtered groups removed. `func` is a predicate that accepts a group and returns `True` to keep that group's values. If `dropna=False`, groups that evaluate to `False` are filled with `NaN`. |
| `g.transform(func, *args, **kwargs)` | Return a DataFrame with the original index. The `func` is passed each group and should return an object with the same dimensions as the group (so the result can be aligned back to the original index). |

In [83]:
# selecting the region if any of the sale record is missing
df0.groupby('region').filter(lambda g: g.sales.isna().any() )

,firm,year,region,sales,profit
id-0,A,2020,East,120.0,15
id-1,A,2021,East,135.0,18
id-2,A,2022,East,135.0,18
id-3,A,2023,East,160.0,25
id-8,C,2020,East,120.0,15
id-9,C,2021,East,NaN,16
id-10,C,2022,East,130.0,19
id-11,C,2023,East,130.0,19


# Aggregate while Keeping Rows

Keyword used in `transform()`

| String | Description |
|---|---|
| 'all' | Returns True for every value if every value is truthy. |
| 'any' | Returns True for every value if any value is truthy. |
| 'count' | Count of non-NA values for the group. |
| 'cumcount' | Number of each item in the group starting at 0 (S). |
| 'cummax' | Cumulative maximum for each group. |
| 'cummin' | Cumulative minimum for each group. |
| 'cumprod' | Cumulative product for each group. |
| 'cumsum' | Cumulative sum for each group. |
| 'diff' | Subtract the previous row from each row. The group needs to be numeric. |
| 'ffill' | Forward fill each group. |
| 'fillna' | Fill in missing values for each group. Must specify method ('ffill' or 'bfill') or value parameter. |
| 'first' | First row for each group. |
| 'mad' | Mean absolute deviation for each group. |
| 'max' | Maximum value for each group. |
| 'mean' | Mean value for each group. |
| 'median' | Mean of each group. |
| 'min' | Minimum value for each group. |
| 'nth' | Nth value for each group. Must specify the n parameter. |
| 'nunique' | Number of unique values for each group. |
| 'pct_change' | Percent change from current row and previous for each group. The group needs to be numeric. |
| 'prod' | Product of each group. |
| 'quantile' | Median of each group. Specify q (0-1) to change the quantile. The group needs to be numeric. |
| 'rank' | Rank of each group. |
| 'sem' | Unbiased standard error of each group. |
| 'shift' | Shift each group row down. Can specify periods (default 1) or freq with date index. |
| 'size' | Size of each group. Only works for a group with a single column (not a dataframe). |
| 'skew' | Skew of each group. |
| 'std' | Standard deviation of each group. |
| 'sum' | Sum of each group. (Will add strings!) |
| 'var' | Variance of each group. |

In [78]:
# create a new column that is mean region sales
df0.assign(region_sales=(df0.groupby('region', observed=True).sales.transform('mean'))).head(3)

,firm,year,region,sales,profit,region_sales
id-0,A,2020,East,120.0,15,132.857143
id-1,A,2021,East,135.0,18,132.857143
id-2,A,2022,East,135.0,18,132.857143


# Melting

- id_vars
- value_vars
- var_name
- value_name

Using `pivot_table` or `groupby` to unmelt.

| Method | Description |
|---|---|
| `.melt(id_vars=, value_vars=, var_name=None, value_name='value', col_level=None, ignore_index=True)` | Return an unpivoted DataFrame: stack each column in `value_vars` into rows, keeping the `id_vars` columns. |

In [85]:
df0.melt(id_vars=['region', 'firm'],value_vars=['profit', 'sales']).sample(5)

,region,firm,variable,value
10,East,C,profit,19.0
15,East,A,sales,160.0
11,East,C,profit,19.0
19,West,B,sales,140.0
17,West,B,sales,110.0


In [87]:
(
    df0
    .melt(id_vars=['region','firm'],
          value_vars=['profit','sales'],
          var_name='metrics',
          value_name='Value')
    .sample(5)
)                      

,region,firm,metrics,Value
3,East,A,profit,25.0
6,West,B,profit,11.0
17,West,B,sales,110.0
16,West,B,sales,90.0
19,West,B,sales,140.0


# flattening hierarchical indexes and columns

In [89]:
def flatten_cols(df):
    cols = ['_'.join(map(str, vals)) for vals in df.columns.to_flat_index()]
    df.columns = cols
    return df

(df0
 .groupby(['region', 'firm', 'year'], observed=True)
 .mean(numeric_only=True).unstack()
 .pipe(flatten_cols))

sales_2020  sales_2021  sales_2022  sales_2023  profit_2020  \
region firm                                                                
East   A          120.0       135.0       135.0       160.0         15.0   
       C          120.0         NaN       130.0       130.0         15.0   
West   B           90.0       110.0       105.0       140.0         10.0   

             profit_2021  profit_2022  profit_2023  
region firm                                         
East   A            18.0         18.0         25.0  
       C            16.0         19.0         19.0  
West   B            12.0         11.0         20.0